# GPU + NPU Roofline Sweep Template

This notebook is a blank slate for custom sweeps.

What is included:
- reusable benchmark helpers
- optional artifact build helper for NPU
- sweep helper functions (GPU, NPU, combined)
- a couple of tiny example plots

What is intentionally not included:
- large default sweeps
- hard-coded analysis assumptions

In [8]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

In [9]:
# ---- Core configuration (edit these first) ----
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
BINARY = REPO_ROOT / "iron" / "operators" / "mul_bench" / "mul_bench_parallel_test"
BENCH_DIR = BINARY.parent

SIZE_GPU = 1 << 20
SIZE_NPU = 1 << 20
ITERS_GPU = 1
ITERS_NPU = 1
REPEATS = 3

NPU_COLUMNS = 8
NPU_CHANNELS = 2
NPU_TILE_SZ = 8192

RUN_GPU = True
RUN_NPU = True

print(f"Binary: {BINARY} (exists: {BINARY.exists()})")

Binary: /home/steven/gpu_bench/IRON/iron/operators/mul_bench/mul_bench_parallel_test (exists: True)


In [ ]:
def artifact_stem(size_npu, r, num_columns=NPU_COLUMNS, num_channels=NPU_CHANNELS, tile_size=NPU_TILE_SZ):
    return f"MulBench_sz{size_npu}_c{num_columns}_ch{num_channels}_t{tile_size}_R{r}_npu2"


def ensure_npu_artifacts(r, size_npu=None):
    """Compile NPU xclbin/bin for (size_npu, r) if missing."""
    if size_npu is None:
        size_npu = SIZE_NPU
    stem = artifact_stem(size_npu, r)
    build_dir = REPO_ROOT / "build"
    xclbin = build_dir / f"{stem}.xclbin"
    insts = build_dir / f"{stem}.bin"

    if xclbin.exists() and insts.exists():
        return

    print(f"Compiling NPU artifacts for R={r}, size={size_npu} ...", flush=True)
    result = subprocess.run(
        [
            sys.executable,
            "iron/operators/mul_bench/run_mul_bench_npu.py",
            "--size", str(size_npu),
            "--r", str(r),
            "--num-columns", str(NPU_COLUMNS),
            "--num-channels", str(NPU_CHANNELS),
            "--tile-size", str(NPU_TILE_SZ),
        ],
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"NPU artifact build failed for R={r}, size={size_npu}\n"
            + (result.stderr[-1000:] if result.stderr else "(no stderr)")
        )


def run_bench(r, size_gpu=None, size_npu=None, iters_gpu=None, iters_npu=None, run_gpu=None, run_npu=None, repeats=None):
    if size_gpu is None:
        size_gpu = SIZE_GPU
    if size_npu is None:
        size_npu = SIZE_NPU
    if iters_gpu is None:
        iters_gpu = ITERS_GPU
    if iters_npu is None:
        iters_npu = ITERS_NPU
    if run_gpu is None:
        run_gpu = RUN_GPU
    if run_npu is None:
        run_npu = RUN_NPU
    if repeats is None:
        repeats = REPEATS

    if not run_gpu and not run_npu:
        raise ValueError("run_gpu and run_npu are both False")

    cmd = [
        str(BINARY),
        "--r", str(r),
        "--size-gpu", str(size_gpu),
        "--size-npu", str(size_npu),
        "--iters-gpu", str(iters_gpu),
        "--iters-npu", str(iters_npu),
        "--repeats", str(repeats),
        "--json",
    ]
    if not run_gpu:
        cmd.append("--no-gpu")
    if not run_npu:
        cmd.append("--no-npu")

    result = subprocess.run(cmd, cwd=str(BENCH_DIR), capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"mul_bench_parallel_test failed (exit={result.returncode})\"
            + (result.stderr[-1000:] if result.stderr else "(no stderr)")
        )

    lines = result.stdout.strip().splitlines()
    if not lines:
        raise RuntimeError("No stdout received from mul_bench_parallel_test")
    return json.loads(lines[-1])

SyntaxError: unterminated f-string literal (detected at line 34) (3553770343.py, line 34)

In [ ]:
def sweep_gpu(r_values, iters_gpu=None, repeats=None):
    points = []
    for i, r in enumerate(r_values, start=1):
        print(f"[GPU {i}/{len(r_values)}] R={r}", flush=True)
        data = run_bench(r, iters_gpu=iters_gpu, run_gpu=True, run_npu=False, repeats=repeats)
        g = data.get("gpu")
        if g:
            points.append((float(g["arith_intensity"]), float(g["tflops"])))
    return points


def sweep_npu(r_values, iters_npu=None, repeats=None, ensure_artifacts=True):
    points = []
    for i, r in enumerate(r_values, start=1):
        print(f"[NPU {i}/{len(r_values)}] R={r}", flush=True)
        if ensure_artifacts:
            ensure_npu_artifacts(r)
        data = run_bench(r, iters_npu=iters_npu, run_gpu=False, run_npu=True, repeats=repeats)
        n = data.get("npu")
        if n:
            points.append((float(n["arith_intensity"]), float(n["tflops"])))
    return points


def sweep_combined(r_values, iters_gpu=None, iters_npu=None, repeats=None, ensure_artifacts=False):
    points = []
    for i, r in enumerate(r_values, start=1):
        print(f"[COMBINED {i}/{len(r_values)}] R={r}", flush=True)
        if ensure_artifacts:
            ensure_npu_artifacts(r)
        data = run_bench(r, iters_gpu=iters_gpu, iters_npu=iters_npu, run_gpu=True, run_npu=True, repeats=repeats)
        ai = r / 2.0
        tf = data.get("combined_tflops")
        points.append((float(ai), float(tf) if tf is not None else None))
    return points

## Run Your Sweep (Edit and Execute)

Start small first, then scale up.

Example:
- set `R_VALUES`
- run one or more sweep helpers
- call a plotting helper

In [11]:
# Example starter (kept small on purpose).
R_VALUES = [1, 4, 8]

# Uncomment what you want to run:
gpu_points = sweep_gpu(R_VALUES, iters_gpu=1, repeats=1)
npu_points = sweep_npu(R_VALUES, iters_npu=1, repeats=1, ensure_artifacts=True)
combined_points = sweep_combined(R_VALUES, iters_gpu=1, iters_npu=1, repeats=1, ensure_artifacts=False)

plot_points(gpu_points, label="GPU", color="royalblue", title="GPU sweep")
plot_three(gpu_points, npu_points, combined_points, title="Quick combined view")

NameError: name 'sweep_gpu' is not defined

In [ ]:
npu_points      = []
# ── NPU sweep ─────────────────────────────────────────────────────────────────
for idx, r in enumerate(R_VALUES):
    print(f"[{idx+1}/{len(R_VALUES)}] R={r:5d}  AI={r/2:.2f} FLOPs/Byte", end="  ", flush=True)
    try:
        if RUN_NPU:
            ensure_npu_artifacts(r)
        datan = run_bench(r, iters_npu=total_batches, run_gpu=False, run_npu=True, repeats=repeats)
        n = datan.get("npu")
        if n:
            npu_points.append((n["arith_intensity"], n["tflops"]))

            print(f"NPU {n['tflops']:.4f} TFLOPS", end="  ", flush=True)
            print(f"NPU BW {n['bandwidth_gbps']:.2f} GB/s")
        else:
            print("no NPU result")
    except Exception as exc:
        print(f"FAILED: {exc}")

print(f"\nCollected {len(npu_points)} NPU points.")